# Dome & Wind Seeing — Building a Constrainable Model

**Question.** Can we build a plausible, physically-grounded model of the **dome/wind
contribution to delivered image quality**, and does the existing data constrain it?

This notebook is the modelling follow-on to
[`Wind_Loading_and_Ingression_Operational_Limits.ipynb`](Wind_Loading_and_Ingression_Operational_Limits.ipynb)
(§16b/§16c), [`Dome_Flow_Estimation.ipynb`](Dome_Flow_Estimation.ipynb) and
[`ESS_Wind_vs_DomePointing.ipynb`](ESS_Wind_vs_DomePointing.ipynb).

## The model we want

Delivered image quality adds in quadrature over independent blur mechanisms:

$$\mathrm{FWHM}^2_{\rm delivered} \;=\; \underbrace{\left(s\cdot\theta_{\rm DIMM}\cdot X^{0.6}\right)^2}_{\text{free atmosphere}} \;+\; \underbrace{\mathrm{FWHM}^2_{\rm inst}}_{\text{optics+detector}} \;+\; \underbrace{\mathrm{FWHM}^2_{\rm dome}}_{\textbf{what we want}}$$

and the dome term should follow near-field optical-turbulence physics, driven by a
**temperature difference** and modulated by **ventilation**:

$$\mathrm{FWHM}_{\rm dome} \;=\; C\,\cdot\,|\Delta T|^{\,p}\,\cdot\,g(V,\Delta)$$

$C_n^2 \propto (\sigma_T/T)^2$ and $\mathrm{FWHM}\propto (C_n^2 L)^{3/5}$, so pure theory
predicts $p = 6/5$. $\Delta T$ is the **mirror-glass − air** temperature difference;
$g(V,\Delta)$ carries the flushing/ventilation dependence on wind speed $V$ and
relative wind angle $\Delta$.

## Headline conclusions

> **1. The wind→IQ effect reported in §16b is not confirmed.** Both supporting arguments are
> artifacts (a *suppressor variable* and a *one-sided clipping bias*), and the effect
> **vanishes within nights** ($\rho = +0.02$) while living entirely between nights
> ($\rho = +0.24$) — the signature of a seasonal/synoptic confound, not a causal
> dome-seeing pathway that should act on minute timescales.
>
> **2. The data does constrain the wind term — as an upper limit.** Within-night, cluster-robust:
> $|k| < 7.2\times10^{-4}\,\mathrm{arcsec/(m/s)^2}$ at 95%, i.e. **< 0.10″ at 12 m/s and
> < 0.29″ at 20 m/s**. This *excludes* the +61%/+63% degradation §16b extrapolated to 20 m/s.
>
> **3. There is a real, robust dome-seeing driver — and it is thermal, not aerodynamic.**
> Mirror–air $|\Delta T|$ survives the within-night test at $t=6.3$ with a clean placebo,
> giving $\mathrm{FWHM}_{\rm dome} \approx 0.10\,|\Delta T|$ arcsec/K. With $|\Delta T|$
> in the model, wind adds **nothing** ($t=0.3$).
>
> **So: sufficient data to falsify the wind model and to detect the thermal one; not yet
> sufficient to calibrate $g(V,\Delta)$.**

## Method: night-level fixed effects + placebo

Every claim below is tested **within nights** (night-demeaned) and validated with a
**placebo** (shuffle the driver inside each night; a true effect survives, an artifact
does not). Standard errors are **clustered on night** — the effective sample size is
~73 nights, not ~36k exposures, and iid errors understate uncertainty by ~8×.

## §1 — Imports & configuration

In [ ]:
import pathlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit, least_squares
from scipy.stats import spearmanr

warnings.filterwarnings("ignore", category=RuntimeWarning)
%matplotlib inline
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

CACHE_DIR = pathlib.Path("../data")
# Full on-sky window: every LSSTCam science night through 2026-07-13 (the final
# night). An earlier revision stopped at 20260701/thermal-20260513 on the mistaken
# belief that science paused in April 2026; it did not — see the wind-loading
# notebook's §2 note. Extending added 43 wind nights and 37 thermal nights.
DAY_START, DAY_END = 20260107, 20260714

WIND_CACHE = CACHE_DIR / f"wind_loading_{DAY_START}_{DAY_END}.parquet"
LOUVER_CACHE = CACHE_DIR / f"wind_loading_louver_{DAY_START}_{DAY_END}.parquet"
# Dated snapshot (223 nights -> 260 nights, now ending 20260713). Pinned by date so
# a re-fetch in ThermalDifferential_PSF_Impact.ipynb cannot silently change this input.
THERMAL_CACHE = CACHE_DIR / "thermal_psf_joined_20260713.parquet"

PIXSCALE = 0.2  # arcsec/pix (LSSTCam)
SIGMA_TO_FWHM = 2.355
KOLMOGOROV_X = 0.6  # seeing airmass exponent: theta ∝ X^0.6

V_CALM = 3.0  # m/s  calm reference
RNG = np.random.default_rng(20260810)

# night-level binning idioms shared with the sibling notebooks
wind_speed_bins = [0, 3, 6, 9, 12, 19]  # top edge > observed max 18.9 m/s
rel_wind_bins = [0, 45, 90, 135, 180]
rel_wind_labels = ["into <45", "45-90", "90-135", "away >135"]

print("caches present:")
for p in (WIND_CACHE, LOUVER_CACHE, THERMAL_CACHE):
    print(f"  {p.name:48s} {'OK' if p.exists() else 'MISSING'}")

## §2 — Load & join

We start from the 25-week wind cache (built by `build_full_cache.py`) and add two joins:

* **louver open/closed** state per exposure (`build_louver.py`) — the §16c ventilation handle;
* **mirror/air thermal** state per exposure from `thermal_psf_joined_20260513.parquet`
  (`../thermal/ThermalDifferential_PSF_Impact.ipynb`), which supplies
  `delta_T_glass_air` — the M1M3 glass-minus-air temperature difference. This is the
  physical driver the dome-seeing model needs and it was **not** used in §16b.

`analysis_ok` is the open-dome science guard from §9 of the wind notebook
(`img_type='science'` ∧ shutter >95% ∧ `can_see_sky` ∧ not vignetted).

In [ ]:
w = pd.read_parquet(WIND_CACHE)
w = w[w["analysis_ok"]].copy()

# ── louver state: time-indexed cache → asof-join on exposure start ───────────
lou = pd.read_parquet(LOUVER_CACHE).reset_index()
lou["time"] = pd.to_datetime(lou["time"]).dt.tz_localize(None)
w = pd.merge_asof(
    w.sort_values("obs_start"),
    lou.sort_values("time"),
    left_on="obs_start",
    right_on="time",
    tolerance=pd.Timedelta("2min"),
    direction="nearest",
)

# ── thermal state: exposure_id join ─────────────────────────────────────────
th = pd.read_parquet(THERMAL_CACHE)
th_cols = [
    "exposure_id",
    "delta_T_glass_air",  # M1M3 glass − air  [K]
    "glass_temp_bulk",
    "glass_temp_spread",
    "inside_temp_mean",
    "inside_temp_spread",
    "outside_temp_301",
]
df = w.merge(th[th_cols], on="exposure_id", how="left")

# ── derived quantities ──────────────────────────────────────────────────────
for c in ["psf_sigma_median", "dimm_seeing", "donut_blur_fwhm", "aos_fwhm"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["psf_fwhm"] = df["psf_sigma_median"] * SIGMA_TO_FWHM * PIXSCALE
df["V"] = df["efd_wind_speed"]
df["V2"] = df["V"] ** 2
# DIMM telemetry has a bad-value tail (max ~40"); Cerro Pachon seeing is ~0.4-2.5".
# Clip to a physical range BEFORE using it, else a few rows dominate any L2 fit.
DIMM_VALID = (0.2, 3.0)
n_bad = int(
    ((df["dimm_seeing"] < DIMM_VALID[0]) | (df["dimm_seeing"] > DIMM_VALID[1])).sum()
)
df.loc[
    (df["dimm_seeing"] < DIMM_VALID[0]) | (df["dimm_seeing"] > DIMM_VALID[1]),
    "dimm_seeing",
] = np.nan
print(
    f"rejected {n_bad} exposures with DIMM outside {DIMM_VALID} arcsec (bad telemetry)"
)

df["atm"] = df["dimm_seeing"] * df["airmass"] ** KOLMOGOROV_X  # DIMM -> line of sight
df["dT"] = df["delta_T_glass_air"]
df["abs_dT"] = df["dT"].abs()
df["dT_inside_outside"] = df["inside_temp_mean"] - df["outside_temp_301"]
df["louver_open"] = df["louver_open"].astype("boolean")

print(
    f"exposures: {len(df):,}   nights: {df['day_obs'].nunique()}   "
    f"day_obs {df['day_obs'].min()} → {df['day_obs'].max()}"
)
print(
    f"wind: median {df['V'].median():.2f}  p95 {df['V'].quantile(0.95):.2f}  "
    f"max {df['V'].max():.2f} m/s"
)

print("\ncolumn coverage:")
for c in [
    "psf_fwhm",
    "donut_blur_fwhm",
    "aos_fwhm",
    "dimm_seeing",
    "seeing_zenith_500nm_median",
    "dT",
    "louver_open",
    "inside_turb_speed",
]:
    n = (
        pd.to_numeric(df[c], errors="coerce").notna().sum()
        if c != "louver_open"
        else int(df[c].notna().sum())
    )
    print(f"  {c:32s} {n:6d} / {len(df)}")

### §2a — Two data limitations to record up front

`seeing_zenith_500nm_median` — the properly zenith-corrected atmospheric reference — is
**100% null** in this window (a known ConsDB gap, see the `reference_consdb_zernikes_null`
note). So our only free-atmosphere proxy is the raw `dimm_seeing`, which we must correct to
the line of sight ourselves via $X^{0.6}$.

The **effective sample size is nights, not exposures**: exposures within a night share
atmosphere, thermal state, wind regime and louver configuration.

In [ ]:
n_exp, n_nights = len(df), df["day_obs"].nunique()
print(
    f"exposures = {n_exp:,}   nights = {n_nights}   "
    f"median exposures/night = {df.groupby('day_obs').size().median():.0f}"
)

# how much wind leverage exists WITHIN a night?
rng_night = df.groupby("day_obs")["V"].agg(lambda s: s.max() - s.min())
print(
    f"\nwithin-night wind range (m/s): median {rng_night.median():.2f}, "
    f"nights >5 m/s: {(rng_night > 5).sum()}/{len(rng_night)}, "
    f">8 m/s: {(rng_night > 8).sum()}"
)

print("\nexposure counts by wind bin (high-wind tail is thin):")
display(
    df.assign(vb=pd.cut(df["V"], wind_speed_bins))
    .groupby("vb", observed=True)
    .agg(
        n=("V", "size"),
        nights=("day_obs", "nunique"),
        n_psf=("psf_fwhm", "count"),
        n_dimm=("dimm_seeing", "count"),
    )
)

## §3 — Re-examining the §16b evidence

§16b concluded that the wind→IQ link is a **near-field/dome-seeing** pathway, resting on two
results. We reproduce both and show each is an artifact of the estimator, not a physical signal.

### §3a — "The PSF–wind partial correlation survives removing DIMM"

This was read as evidence of a pathway independent of the free atmosphere. But at Cerro
Pachón **DIMM seeing falls as wind rises** ($\rho\simeq-0.37$): windy nights are
better-mixed, not worse. A control that is *negatively* correlated with the predictor and
*positively* with the response is a **suppressor variable** — partialling it out is
guaranteed to *inflate* the correlation:

$$r_{yx\cdot z}=\frac{r_{yx}-r_{yz}r_{xz}}{\sqrt{(1-r_{yz}^2)(1-r_{xz}^2)}}\;>\;r_{yx}\quad\text{whenever } r_{yz}r_{xz}<0 .$$

So "survival" here carries **no evidential weight** — it is arithmetic.

In [ ]:
d = df.dropna(subset=["psf_fwhm", "V2", "dimm_seeing", "airmass"]).copy()


def pear(a, b):
    return np.corrcoef(a, b)[0, 1]


r_yx = pear(d["V2"], d["psf_fwhm"])  # predictor  ↔ response
r_yz = pear(d["dimm_seeing"], d["psf_fwhm"])
r_xz = pear(d["V2"], d["dimm_seeing"])  # predictor  ↔ "control"

partial = (r_yx - r_yz * r_xz) / np.sqrt((1 - r_yz**2) * (1 - r_xz**2))

print(f"n = {len(d):,} exposures, {d['day_obs'].nunique()} nights\n")
print(
    f"  rho_spearman(V, DIMM)        = {spearmanr(d['V'], d['dimm_seeing'])[0]:+.3f}"
    "   <-- NEGATIVE: windy nights have BETTER outside seeing"
)
print(f"  r(V^2, PSF)                  = {r_yx:+.3f}")
print(f"  r(DIMM, PSF)                 = {r_yz:+.3f}")
print(
    f"  r(V^2, DIMM)                 = {r_xz:+.3f}   <-- suppressor (sign is negative)"
)
print(
    f"\n  partial r(V^2, PSF | DIMM)   = {partial:+.3f}"
    f"  vs raw {r_yx:+.3f}  ->  INFLATED by construction"
)

print(
    f"\n  DIMM median: calm (V<3) = {d.loc[d['V'] < 3, 'dimm_seeing'].median():.3f}\"  "
    f"windy (V>12) = {d.loc[d['V'] > 12, 'dimm_seeing'].median():.3f}\""
)

### §3b — "Quadrature-subtracting the atmosphere leaves a wind-dependent residual"

The residual $\sqrt{\mathrm{PSF}^2-\theta_{\rm atm}^2}$ correlates with wind at $r\approx+0.46$,
which looks like a dome-seeing detection. Two problems compound:

1. **One-sided clipping.** $\theta_{\rm DIMM}X^{0.6}$ is *systematically larger* than the
   delivered PSF (median 1.05″ vs 1.01″) — different wavelength, aperture and integration
   time mean the DIMM-to-PSF scale $s$ is not 1. So the difference goes negative for **54%**
   of exposures and is clipped at zero. Clipping at zero manufactures a strictly positive,
   heteroscedastic residual out of what is mostly noise plus a calibration offset.
2. **The subtrahend itself depends on wind** (§3a): higher wind → lower DIMM → smaller thing
   subtracted → larger residual. The "signal" is partly just $-\theta_{\rm atm}(V)$.

The **placebo** settles it: shuffle wind *within each night*, destroying any real
wind→IQ link while preserving all distributions. A genuine effect must collapse to zero.

In [ ]:
d = d.copy()
d["resid_fwhm"] = np.sqrt(np.clip(d["psf_fwhm"] ** 2 - d["atm"] ** 2, 0, None))
frac_clipped = (d["psf_fwhm"] ** 2 < d["atm"] ** 2).mean()

# placebo: permute wind inside each night
d["V_placebo"] = d.groupby("day_obs")["V"].transform(
    lambda s: RNG.permutation(s.values)
)

print(f"fraction clipped to zero      : {100 * frac_clipped:.0f}%")
print(
    f"median atm - median PSF       : {d['atm'].median() - d['psf_fwhm'].median():+.3f}\""
    "   <-- DIMM overpredicts -> subtraction goes negative"
)
print()
print(
    f"  r(V,          resid_fwhm)   = {pear(d['V'], d['resid_fwhm']):+.3f}"
    "   <-- 'detection'"
)
print(
    f"  r(V_placebo,  resid_fwhm)   = {pear(d['V_placebo'], d['resid_fwhm']):+.3f}"
    "   <-- PLACEBO, should be ~0"
)
print(
    f"  r(V,          atm)          = {pear(d['V'], d['atm']):+.3f}"
    "   <-- subtrahend shrinks with wind"
)
print("\n=> the placebo reproduces most of the 'signal' => artifact, not physics.")

In [ ]:
# DIMM as a PSF predictor at three aggregation scales — why per-exposure subtraction fails
d20 = (
    d.set_index("obs_start")
    .groupby("day_obs")[["psf_fwhm", "atm"]]
    .resample("20min")
    .median()
    .dropna()
)
nightly = d.groupby("day_obs")[["psf_fwhm", "atm"]].median().dropna()

print("r(atm, PSF) by aggregation scale:")
print(f"  per-exposure          {pear(d['atm'], d['psf_fwhm']):+.3f}   (n={len(d):,})")
print(
    f"  20-min blocks         {pear(d20['atm'], d20['psf_fwhm']):+.3f}   (n={len(d20):,})"
)
print(
    f"  night medians         {pear(nightly['atm'], nightly['psf_fwhm']):+.3f}   (n={len(nightly)})"
)
print(
    f"\nmedian atm = {d['atm'].median():.3f}\" vs median PSF = {d['psf_fwhm'].median():.3f}\""
)
print(
    "  -> DIMM OVERPREDICTS the delivered PSF, so PSF^2 - atm^2 is negative ~half the"
)
print(
    "     time. Correlation also improves with averaging, so per-exposure DIMM carries"
)
print("     extra noise. Both reasons say: fit the budget jointly, never subtract.")

### §3c — The decisive test: between-night vs within-night

Dome seeing is a *local, fast* mechanism — turbulent air in the light path responds to wind
within minutes. So a causal wind→IQ effect **must** appear when we compare exposures
*inside the same night*. A correlation that exists only *between* nights instead indicates
that windy nights are meteorologically different nights (season, synoptic regime, humidity).

We split each quantity into its night-median (between) and night-demeaned (within) parts.

In [ ]:
def add_within(frame, cols, group="day_obs"):
    # night-demean `cols` (subtract per-night median) -> '<col>_dm'
    out = frame.copy()
    for c in cols:
        out[f"{c}_dm"] = out[c] - out.groupby(group)[c].transform("median")
    return out


IQ = ["psf_fwhm", "donut_blur_fwhm"]
base = df.dropna(subset=["psf_fwhm", "V", "airmass"]).copy()
base = add_within(base, ["psf_fwhm", "V", "V2", "airmass"])

rows = []
for col in IQ + ["dimm_seeing"]:
    need = list(
        dict.fromkeys([col, "V2", "dimm_seeing", "airmass"])
    )  # dedup: col may BE dimm
    sub = df.dropna(subset=need).copy()
    sub = add_within(sub, need)
    nm = sub.groupby("day_obs")[need].median()
    rows.append(
        dict(
            metric=col,
            n_exp=len(sub),
            n_nights=len(nm),
            between_V2=pear(nm["V2"], nm[col]),
            within_V2=pear(sub["V2_dm"], sub[f"{col}_dm"]),
            within_DIMM=pear(sub["dimm_seeing_dm"], sub[f"{col}_dm"]),
            within_airmass=pear(sub["airmass_dm"], sub[f"{col}_dm"]),
        )
    )
tbl = pd.DataFrame(rows).set_index("metric")
display(tbl.round(3))

print("The wind effect is ~10x weaker within nights than between nights.")
print("Crucially the SAME within-night contrast recovers airmass and DIMM cleanly,")
print("so the design is NOT underpowered -- the wind signal is genuinely absent.")

In [ ]:
# variance decomposition + robustness of the within-night null
tot = base["psf_fwhm"].var()
btw = base.groupby("day_obs")["psf_fwhm"].median().reindex(base["day_obs"]).values.var()
print(
    f"PSF variance: total {tot:.4f} | between-night {btw:.4f} ({100*btw/tot:.0f}%) "
    f"| within-night {tot-btw:.4f} ({100*(tot-btw)/tot:.0f}%)"
)

# (i) smoothing wind (dome response could lag the anemometer)
print("\nwithin-night rho(V_dm, PSF_dm), wind time-averaged over:")
tmp = base.sort_values("obs_start").set_index("obs_start")
for win in ["5min", "15min", "30min", "60min", "120min"]:
    sm = tmp.groupby("day_obs")["V"].transform(lambda s: s.rolling(win).mean())
    sm_dm = sm - sm.groupby(tmp["day_obs"]).transform("median")
    m = sm_dm.notna() & tmp["psf_fwhm_dm"].notna()
    print(f"   {win:>7s}  rho = {spearmanr(sm_dm[m], tmp['psf_fwhm_dm'][m])[0]:+.3f}")

# (ii) restrict to nights that actually span a wide wind range
hi = rng_night[rng_night > 5].index
dh = base[base["day_obs"].isin(hi)]
print(
    f"\nnights with wind range >5 m/s ({len(hi)} nights, {len(dh):,} exp): "
    f"rho(V_dm, PSF_dm) = {spearmanr(dh['V_dm'], dh['psf_fwhm_dm'])[0]:+.3f}"
)

# (iii) seasonal confound at night level
nm = (
    df.groupby("day_obs")
    .agg(V=("V", "median"), psf=("psf_fwhm", "median"))
    .reset_index()
)
nm["doy"] = pd.to_datetime(nm["day_obs"], format="%Y%m%d").dt.dayofyear
print(
    f"\nseasonal confound: rho(day-of-year, night wind) = "
    f"{spearmanr(nm['doy'], nm['V'])[0]:+.3f}  "
    "-> wind is seasonally trended, so between-night wind is confounded"
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

# (a) between-night
nmp = (
    df.groupby("day_obs")
    .agg(V=("V", "median"), psf=("psf_fwhm", "median"), n=("psf_fwhm", "size"))
    .dropna()
)
axes[0].scatter(
    nmp["V"],
    nmp["psf"],
    s=18 + nmp["n"] / 25,
    alpha=0.7,
    color="tomato",
    edgecolor="k",
    linewidth=0.3,
)
p = np.polyfit(nmp["V"], nmp["psf"], 1)
xs = np.linspace(nmp["V"].min(), nmp["V"].max(), 50)
axes[0].plot(xs, np.polyval(p, xs), "k--", lw=1.5)
axes[0].set(
    xlabel="night median wind [m/s]",
    ylabel='night median PSF FWHM ["]',
    title=f"(a) BETWEEN nights (n={len(nmp)})\nr = {pear(nmp['V'], nmp['psf']):+.3f}"
    "  — looks like an effect",
)

# (b) within-night, binned
b = base.copy()
b["vb"] = pd.cut(b["V"], np.arange(0, 16, 2))
g = b.groupby("vb", observed=True).agg(
    x=("V", "median"),
    y=("psf_fwhm_dm", "median"),
    e=("psf_fwhm_dm", lambda s: s.std() / max(np.sqrt(len(s)), 1)),
    n=("psf_fwhm_dm", "size"),
)
axes[1].errorbar(g["x"], g["y"], yerr=g["e"], fmt="o-", color="steelblue", capsize=3)
axes[1].axhline(0, color="k", lw=0.8, ls=":")
axes[1].set(
    xlabel="wind speed [m/s]",
    ylabel='night-demeaned PSF FWHM ["]',
    title=f"(b) WITHIN nights (n={len(b):,})\n"
    f"r = {pear(b['V2_dm'], b['psf_fwhm_dm']):+.3f}  — flat, no effect",
)

# (c) placebo comparison on the quadrature residual
axes[2].bar(
    ["real\nwind", "placebo\n(shuffled\nin night)"],
    [pear(d["V"], d["resid_fwhm"]), pear(d["V_placebo"], d["resid_fwhm"])],
    color=["tomato", "grey"],
    edgecolor="k",
)
axes[2].axhline(0, color="k", lw=0.8)
axes[2].set(
    ylabel=r"r(wind, $\sqrt{PSF^2-atm^2}$)",
    title="(c) quadrature residual\nplacebo reproduces the 'signal'",
)

for a in axes:
    a.grid(alpha=0.3)
fig.suptitle(
    "§3 — The wind→IQ correlation lives only between nights",
    y=1.02,
    fontsize=13,
    weight="bold",
)
fig.tight_layout()
plt.show()

## §4 — How large a wind term is still allowed?

The within-night null is not "no information" — it is an **upper limit**, and that is a
usable model constraint. We fit, within nights,

$$\mathrm{PSF}_{\rm dm} = a + k\,V^2_{\rm dm}$$

with standard errors **clustered on night** (exposures in a night are far from independent;
iid errors understate the uncertainty here by ~8×).

In [ ]:
def cluster_ols(frame, y, xcols, cluster="day_obs"):
    # OLS with cluster-robust (sandwich) SEs; returns a tidy DataFrame
    dd = frame.dropna(subset=[y] + xcols + [cluster])
    X = np.column_stack([np.ones(len(dd))] + [dd[c].to_numpy(float) for c in xcols])
    yv = dd[y].to_numpy(float)
    beta, *_ = np.linalg.lstsq(X, yv, rcond=None)
    resid = yv - X @ beta
    XtXi = np.linalg.pinv(X.T @ X)
    meat = np.zeros((X.shape[1],) * 2)
    for g in dd[cluster].unique():
        m = (dd[cluster] == g).to_numpy()
        u = X[m].T @ resid[m]
        meat += np.outer(u, u)
    se_cl = np.sqrt(np.diag(XtXi @ meat @ XtXi))
    s2 = resid @ resid / (len(dd) - X.shape[1])
    se_iid = np.sqrt(np.diag(s2 * XtXi))
    names = ["const"] + xcols
    return (
        pd.DataFrame(
            {
                "coef": beta,
                "se_cluster": se_cl,
                "se_iid": se_iid,
                "t_cluster": beta / se_cl,
            },
            index=names,
        ),
        len(dd),
        dd[cluster].nunique(),
    )


res, n, ng = cluster_ols(base, "psf_fwhm_dm", ["V2_dm"])
display(res.round(6))
print(f"n = {n:,} exposures across {ng} nights")

k, se_k = res.loc["V2_dm", "coef"], res.loc["V2_dm", "se_cluster"]
print(
    f"\nSE inflation from clustering: {res.loc['V2_dm','se_cluster']/res.loc['V2_dm','se_iid']:.1f}x"
)

k_lim = 1.96 * se_k
print(
    f"\nk = {k:+.6f} +/- {se_k:.6f}  ->  95% upper limit |k| < {k_lim:.2e} arcsec/(m/s)^2"
)
print("\nexcluded dome-seeing FWHM contribution (linear-in-V^2 form):")
for V in [10, 12, 15, 20]:
    print(f'   V = {V:4.1f} m/s :  < {k_lim * V**2:.3f}"')

In [ ]:
# contrast with the between-night slope that §16b effectively used
nb_ = df.groupby("day_obs").agg(psf=("psf_fwhm", "median"), V=("V", "median")).dropna()
Xb = np.column_stack([np.ones(len(nb_)), nb_["V"] ** 2])
bb, *_ = np.linalg.lstsq(Xb, nb_["psf"].to_numpy(), rcond=None)
rb = nb_["psf"].to_numpy() - Xb @ bb
se_b = np.sqrt(np.diag((rb @ rb / (len(nb_) - 2)) * np.linalg.pinv(Xb.T @ Xb)))

print(f"BETWEEN-night  k = {bb[1]:+.6f} +/- {se_b[1]:.6f}   (n={len(nb_)} nights)")
print(f"WITHIN-night   k = {k:+.6f} +/- {se_k:.6f}   (95% limit {k_lim:.2e})")
print(f'\nat 20 m/s:  between-night implies {bb[1]*400:.2f}" of excess,')
print(f'            within-night allows at most {k_lim*400:.2f}"')
print(f"  -> inconsistent by {abs(bb[1])/k_lim:.1f}x")
print("\n§16b's +61%/+63% degradation at 20 m/s is EXCLUDED by the within-night limit.")

## §5 — §16c revisited: the louver contrast is a night-level confound

§16c reported that with louvers **closed** the wind→PSF slope is ~4× steeper, and read this
as louvers *mitigating* dome seeing. We reproduce the slope contrast, then check what the
comparison is actually made of: louver state is essentially a **per-night configuration**,
so an open-vs-closed contrast is a between-night comparison — exactly the confound §3c
identified.

In [ ]:
dl = df.dropna(subset=["psf_fwhm", "V", "louver_open"]).copy()
dl["lopen"] = dl["louver_open"].astype(bool)


def fit_k(g):
    X = np.column_stack([np.ones(len(g)), g["V"] ** 2])
    b, *_ = np.linalg.lstsq(X, g["psf_fwhm"].to_numpy(), rcond=None)
    return b


print("pooled a + k*V^2 by louver state (reproduces the §16c headline):")
for lab, g in [("OPEN", dl[dl["lopen"]]), ("CLOSED", dl[~dl["lopen"]])]:
    a, kk = fit_k(g)
    print(
        f'  {lab:6s} a={a:.3f}  k={kk:.5f}  PSF@12m/s={a + kk*144:.3f}"  '
        f"n={len(g):6,}  nights={g['day_obs'].nunique():2d}  V_med={g['V'].median():.2f}"
    )

frac = dl.groupby("day_obs")["lopen"].mean()
print(
    f"\nper-night louver open fraction: all-open(>0.9) {int((frac > 0.9).sum())} | "
    f"all-closed(<0.1) {int((frac < 0.1).sum())} | mixed {int(((frac >= 0.1) & (frac <= 0.9)).sum())}"
    f"  (of {len(frac)} nights)"
)
print(
    "=> louver state is a NIGHT-LEVEL label: the 'closed' arm is only a handful of nights,"
)
print("   so the contrast is between-night, not a controlled ventilation experiment.")

In [ ]:
# the only valid version of the test: nights that actually switched state
mixed = frac[(frac >= 0.1) & (frac <= 0.9)].index
dm = dl[dl["day_obs"].isin(mixed)].copy()
dm = add_within(dm, ["psf_fwhm"])
dm["vb"] = pd.cut(dm["V"], wind_speed_bins)

print(f"WITHIN mixed nights ({len(mixed)} nights, {len(dm):,} exposures)")
print("night-demeaned PSF FWHM median by louver state and matched wind bin:")
piv = dm.pivot_table(
    index="vb", columns="lopen", values="psf_fwhm_dm", aggfunc="median", observed=True
)
cnt = dm.pivot_table(
    index="vb", columns="lopen", values="psf_fwhm_dm", aggfunc="size", observed=True
)
piv.columns = ["closed", "open"]
cnt.columns = ["n_closed", "n_open"]
display(pd.concat([piv.round(3), cnt], axis=1))
print('No monotonic wind trend and differences ~0.00-0.05" -> the 4x slope contrast')
print("does not reproduce once night is controlled for.")

## §6 — What *does* drive dome seeing: mirror–air $\Delta T$

Now we test the physically-motivated driver. Near-field optical turbulence is set by
**temperature inhomogeneity**, not by wind *per se* — wind only modulates how fast that
inhomogeneity is flushed away. The relevant quantity is the **mirror-glass minus air**
temperature difference, `delta_T_glass_air`, which §16b never used.

The sign should not matter (either a warm or a cold mirror drives convection), so we test
$|\Delta T|$.

In [ ]:
th_df = df.dropna(subset=["psf_fwhm", "abs_dT", "V"]).copy()
th_df = add_within(th_df, ["psf_fwhm", "abs_dT", "V2", "dT"])

print(f"n = {len(th_df):,} exposures, {th_df['day_obs'].nunique()} nights")
print(
    f"|dT_glass_air|: median {th_df['abs_dT'].median():.3f} K, "
    f"p95 {th_df['abs_dT'].quantile(0.95):.3f} K\n"
)

cands = {
    "abs_dT": "|mirror-air dT| [K]",
    "dT": "signed mirror-air dT [K]",
    "glass_temp_spread": "glass temp spread [K]",
    "inside_temp_spread": "inside air temp spread [K]",
    "dT_inside_outside": "inside-outside dT [K]",
    "inside_turb_speed": "inside turbulence speed [m/s]",
    "V2": "wind V^2 [(m/s)^2]",
}
rows = []
for c, lab in cands.items():
    s = df.dropna(subset=["psf_fwhm", c]).copy()
    if len(s) < 500:
        continue
    s = add_within(s, ["psf_fwhm", c])
    nmx = s.groupby("day_obs")[["psf_fwhm", c]].median()
    rows.append(
        dict(
            driver=lab,
            n=len(s),
            nights=len(nmx),
            pooled=spearmanr(s[c], s["psf_fwhm"])[0],
            between=spearmanr(nmx[c], nmx["psf_fwhm"])[0],
            within=spearmanr(s[f"{c}_dm"], s["psf_fwhm_dm"])[0],
        )
    )
cand_tbl = (
    pd.DataFrame(rows)
    .set_index("driver")
    .sort_values("within", key=abs, ascending=False)
)
display(cand_tbl.round(3))
print(
    "|mirror-air dT| is the ONLY candidate with a substantial WITHIN-night correlation."
)

In [ ]:
# quantify, with placebo and cluster-robust SEs
res_dT, n_dT, ng_dT = cluster_ols(th_df, "psf_fwhm_dm", ["abs_dT_dm"])
display(res_dT.round(5))

b_dT = res_dT.loc["abs_dT_dm", "coef"]
print(f"n = {n_dT:,} exposures, {ng_dT} nights")
print(
    f"\nFWHM_dome ~ {b_dT:.3f} arcsec per K of |mirror-air dT|   "
    f"(t = {res_dT.loc['abs_dT_dm','t_cluster']:+.2f})"
)

# placebo
th_df["abs_dT_placebo"] = th_df.groupby("day_obs")["abs_dT"].transform(
    lambda s: RNG.permutation(s.values)
)
th_df["abs_dT_placebo_dm"] = th_df["abs_dT_placebo"] - th_df.groupby("day_obs")[
    "abs_dT_placebo"
].transform("median")
print(
    f"\n  real     rho(|dT|_dm, PSF_dm) = {spearmanr(th_df['abs_dT_dm'], th_df['psf_fwhm_dm'])[0]:+.3f}"
)
print(
    f"  PLACEBO  rho                  = {spearmanr(th_df['abs_dT_placebo_dm'], th_df['psf_fwhm_dm'])[0]:+.3f}"
    "   <-- clean null: this is a REAL effect"
)

# joint fit: does wind add anything once thermal is in?
res_j, n_j, _ = cluster_ols(th_df, "psf_fwhm_dm", ["abs_dT_dm", "V2_dm"])
print("\njoint within-night fit  PSF_dm ~ |dT|_dm + V^2_dm :")
display(res_j.round(6))
print("thermal is highly significant; wind contributes nothing once dT is controlled.")

In [ ]:
# functional form: excess = C |dT|^p   (theory: Cn^2 ~ dT^2 -> FWHM ~ dT^(6/5))
th_df["q"] = pd.qcut(th_df["abs_dT"], 12, duplicates="drop")
gb = th_df.groupby("q", observed=True).agg(
    x=("abs_dT", "median"), y=("psf_fwhm_dm", "median"), n=("psf_fwhm_dm", "size")
)
gb["y"] -= gb["y"].min()

popt, pcov = curve_fit(
    lambda x, C, p: C * x**p, gb["x"], gb["y"], p0=[0.05, 1.2], maxfev=40000
)
perr = np.sqrt(np.diag(pcov))
print(
    f"fit: excess = {popt[0]:.4f} * |dT|^{popt[1]:.2f}  (+/- {perr[1]:.2f} on exponent)"
)
print(f"theory (Cn^2 ~ dT^2, FWHM ~ (Cn^2 L)^3/5) predicts p = {6/5:.1f}")
print(
    f"-> measured p = {popt[1]:.2f} is SHALLOWER than theory: consistent with the sensor"
)
print(
    "   dT being an imperfect proxy for the dT actually in the light path (attenuation"
)
print("   toward p<1 is the classic errors-in-variables signature).")

# ventilation-mitigation test: is the dT slope weaker at higher wind?
print("\nventilation-mitigation test — within-night |dT| slope by wind tercile:")
th_df["vt"] = pd.qcut(th_df["V"], 3, labels=["low V", "mid V", "high V"])
for lab, s in th_df.groupby("vt", observed=True):
    r_, _, _ = cluster_ols(s, "psf_fwhm_dm", ["abs_dT_dm"])
    print(
        f"   {lab:7s} V_med={s['V'].median():5.2f} m/s   "
        f"b = {r_.loc['abs_dT_dm','coef']:+.4f} +/- {r_.loc['abs_dT_dm','se_cluster']:.4f} \"/K"
        f"   (t={r_.loc['abs_dT_dm','t_cluster']:+.2f})"
    )
res_i, _, _ = cluster_ols(
    th_df.assign(
        inter=(
            th_df["abs_dT_dm"] * th_df["V_dm"]
            if "V_dm" in th_df
            else th_df["abs_dT_dm"]
            * (th_df["V"] - th_df.groupby("day_obs")["V"].transform("median"))
        )
    ),
    "psf_fwhm_dm",
    ["abs_dT_dm", "inter"],
)
print(
    f"\ninteraction |dT| x V : {res_i.loc['inter','coef']:+.5f} +/- "
    f"{res_i.loc['inter','se_cluster']:.5f} (t={res_i.loc['inter','t_cluster']:+.2f})"
)
print("-> no significant mitigation detected: g(V) is NOT yet constrained.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

# (a) within-night PSF excess vs |dT|
gp = th_df.copy()
gp["ab"] = pd.cut(gp["abs_dT"], [0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.5, 3.0])
g1 = gp.groupby("ab", observed=True).agg(
    x=("abs_dT", "median"),
    y=("psf_fwhm_dm", "median"),
    e=("psf_fwhm_dm", lambda s: s.std() / max(np.sqrt(len(s)), 1)),
    n=("psf_fwhm_dm", "size"),
)
axes[0].errorbar(g1["x"], g1["y"], yerr=g1["e"], fmt="o-", color="firebrick", capsize=3)
xs = np.linspace(0.02, 2.0, 100)
axes[0].plot(
    xs,
    popt[0] * xs ** popt[1] + g1["y"].iloc[0] - popt[0] * g1["x"].iloc[0] ** popt[1],
    "k--",
    lw=1.2,
    label=f"$C|\\Delta T|^{{{popt[1]:.2f}}}$",
)
axes[0].axhline(0, color="k", lw=0.8, ls=":")
axes[0].set(
    xlabel="|mirror $-$ air $\\Delta T$| [K]",
    ylabel='night-demeaned PSF FWHM ["]',
    title=f"(a) THERMAL driver — real\nwithin-night $t$ = "
    f"{res_dT.loc['abs_dT_dm','t_cluster']:+.1f}",
)
axes[0].legend()
axes[0].grid(alpha=0.3)

# (b) real vs placebo, thermal and wind
labels = ["|dT|\nreal", "|dT|\nplacebo", "$V^2$\nreal", "$V^2$\nplacebo"]
th_df["V2_placebo"] = th_df.groupby("day_obs")["V2"].transform(
    lambda s: RNG.permutation(s.values)
)
th_df["V2_placebo_dm"] = th_df["V2_placebo"] - th_df.groupby("day_obs")[
    "V2_placebo"
].transform("median")
vals = [
    spearmanr(th_df["abs_dT_dm"], th_df["psf_fwhm_dm"])[0],
    spearmanr(th_df["abs_dT_placebo_dm"], th_df["psf_fwhm_dm"])[0],
    spearmanr(th_df["V2_dm"], th_df["psf_fwhm_dm"])[0],
    spearmanr(th_df["V2_placebo_dm"], th_df["psf_fwhm_dm"])[0],
]
axes[1].bar(
    labels,
    vals,
    color=["firebrick", "lightgrey", "steelblue", "lightgrey"],
    edgecolor="k",
)
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set(
    ylabel=r"within-night $\rho$ with PSF",
    title="(b) placebo test\nthermal survives, wind does not",
)
axes[1].grid(alpha=0.3, axis="y")

# (c) allowed vs claimed wind contribution
Vg = np.linspace(0, 20, 100)
axes[2].fill_between(
    Vg,
    0,
    k_lim * Vg**2,
    color="steelblue",
    alpha=0.3,
    label="allowed by within-night limit (95%)",
)
axes[2].plot(Vg, bb[1] * Vg**2, "r--", lw=2, label="§16b between-night extrapolation")
axes[2].axvline(
    df["V"].max(), color="grey", ls=":", label=f"max observed {df['V'].max():.1f} m/s"
)
axes[2].set(
    xlabel="wind speed [m/s]",
    ylabel='dome-seeing FWHM contribution ["]',
    title="(c) wind term: upper limit vs claim",
)
axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3)

fig.suptitle(
    "§6 — The dome-seeing driver is thermal, not aerodynamic",
    y=1.02,
    fontsize=13,
    weight="bold",
)
fig.tight_layout()
plt.show()

## §7 — Assembling the model

We now fit the full variance budget **jointly**, never by subtraction:

$$\mathrm{FWHM}^2 = (s\,\theta_{\rm DIMM}X^{0.6})^2 + \mathrm{FWHM}_{\rm inst}^2 + \left(C\,|\Delta T|^{p}\right)^2 + \left(k_{\rm eff}V^2\right)^2$$

Two cautions drive the implementation:

* **`s` is degenerate with the instrument floor.** Left free, the fit prefers
  $s\approx0.57$ with a large constant $\mathrm{FWHM}_{\rm inst}\approx0.75''$ — it trades
  atmospheric scaling against a fixed pedestal, because both are nearly flat in the drivers
  we have. Fixing $s=1$ (DIMM *defines* the free-atmosphere term) instead pushes
  $\mathrm{FWHM}_{\rm inst}\to0$. We report both: the dome coefficient $C$ is what we care
  about, and it is stable at $0.18-0.25''/\mathrm{K}^{p}$ across this degeneracy.
* **The wind term is fit as a limit,** not a measurement, since §4 shows it is unresolved.

In [ ]:
fit_df = th_df.dropna(subset=["psf_fwhm", "atm", "abs_dT", "V"]).copy()
print(
    f"joint-fit sample: {len(fit_df):,} exposures, {fit_df['day_obs'].nunique()} nights"
)


def model_fwhm(theta, dd, s_fixed=None, p_fixed=None):
    i = 0
    s = s_fixed if s_fixed is not None else theta[i]
    i += 0 if s_fixed is not None else 1
    inst, C = theta[i], theta[i + 1]
    i += 2
    p = p_fixed if p_fixed is not None else theta[i]
    i += 0 if p_fixed is not None else 1
    k = theta[i]
    var = (
        (s * dd["atm"]) ** 2
        + inst**2
        + (C * dd["abs_dT"].clip(lower=1e-3) ** p) ** 2
        + (k * dd["V"] ** 2) ** 2
    )
    return np.sqrt(np.clip(var, 1e-9, None))


def run_fit(s_fixed, p_fixed, label):
    th0 = [0.35, 0.10, 1e-4]
    if s_fixed is None:
        th0 = [0.9] + th0
    if p_fixed is None:
        th0 = th0[:-1] + [1.0, 1e-4]
    r = least_squares(
        lambda t: model_fwhm(t, fit_df, s_fixed, p_fixed) - fit_df["psf_fwhm"],
        th0,
        x_scale="jac",
        loss="soft_l1",
        f_scale=0.1,
    )
    rms = np.sqrt(np.mean(r.fun**2))
    mad = np.median(np.abs(r.fun))
    names = (
        ([] if s_fixed is not None else ["s"])
        + ["inst", "C"]
        + ([] if p_fixed is not None else ["p"])
        + ["k"]
    )
    out = dict(zip(names, r.x))
    out.update(
        s=s_fixed if s_fixed is not None else out.get("s"),
        p=p_fixed if p_fixed is not None else out.get("p"),
        rms=rms,
        mad=mad,
        label=label,
    )
    return out


fits = [
    run_fit(None, 1.2, "s FREE, p=6/5 (theory)"),
    run_fit(1.0, 1.2, "s=1 (prior), p=6/5 (theory)"),
    run_fit(1.0, None, "s=1 (prior), p free"),
    run_fit(1.0, 0.5, "s=1 (prior), p=0.5 (empirical)"),
]
ft = pd.DataFrame(fits)[["label", "s", "inst", "C", "p", "k", "rms", "mad"]]
display(ft.round(4))
print(
    'Row 1 (s free) prefers s~0.57 with a ~0.75" pedestal; rows 2-4 (s=1) push inst -> 0.'
)
print(
    "s and inst are DEGENERATE. Note C stays in 0.18-0.25 across all rows -> the dome"
)
print("term is robust to that degeneracy, which is what matters for the model.")

In [ ]:
# adopted model + budget table
best = fits[3]  # s=1, empirical exponent
C_ad, p_ad, inst_ad = best["C"], best["p"], best["inst"]
print("ADOPTED DOME-SEEING MODEL")
print("=" * 64)
print(f"  FWHM_dome = {C_ad:.3f} * |dT_mirror-air|^{p_ad:.2f}   [arcsec, dT in K]")
print(f'  FWHM_inst = {inst_ad:.3f}"      (optics + detector floor)')
print(f"  wind term : UNRESOLVED, |k| < {k_lim:.2e} arcsec/(m/s)^2  (95%)")
print(f"  g(V, Delta): NOT CONSTRAINED — no significant V or Delta dependence detected")
print("=" * 64)

budget = pd.DataFrame(
    {
        "|dT| [K]": [0.1, 0.25, 0.5, 1.0, 2.0],
    }
)
budget['FWHM_dome ["]'] = C_ad * budget["|dT| [K]"] ** p_ad
budget['as % of 1.0" total'] = 100 * budget['FWHM_dome ["]'] / 1.0
print("\nthermal dome-seeing budget:")
display(budget.round(3))

print("wind contribution ceiling (95% upper limit — may be zero):")
wb = pd.DataFrame({"V [m/s]": [6, 10, 12, 15, 20]})
wb['FWHM_dome max ["]'] = k_lim * wb["V [m/s]"] ** 2
display(wb.round(3))

## §8 — Is the data sufficient?

**Sufficient to falsify** the large wind-driven dome-seeing term: yes. The within-night
limit excludes contributions above 0.10″ at 12 m/s / 0.29″ at 20 m/s, ruling out the
§16b extrapolation.

**Sufficient to detect** a thermal dome-seeing term: yes — $t=6.3$ with a clean placebo,
$\mathrm{FWHM}_{\rm dome}\approx0.10\,|\Delta T|$ arcsec/K.

**Sufficient to calibrate the full model** $C|\Delta T|^p g(V,\Delta)$: **no.** The binding
limitations are below.

In [ ]:
lims = pd.DataFrame(
    [
        dict(
            limitation="Effective N is nights, not exposures",
            detail=f"{n_exp:,} exposures but only {n_nights} nights; "
            "clustering inflates SEs ~8x",
            blocks="all between-night inference",
        ),
        dict(
            limitation="High-wind tail is thin",
            detail=f"only {int((df['V'] > 15).sum())} exposures >15 m/s; max "
            f"{df['V'].max():.1f} m/s",
            blocks="any 20 m/s operational limit (always extrapolation)",
        ),
        dict(
            limitation="seeing_zenith_500nm is 100% null",
            detail="no properly zenith-corrected atmospheric reference in window",
            blocks="clean separation of atm vs dome",
        ),
        dict(
            limitation="DIMM-to-PSF scale s is degenerate",
            detail=f"DIMM overpredicts PSF (median {d['atm'].median():.2f}\" vs "
            f"{d['psf_fwhm'].median():.2f}\"); s trades off against inst floor",
            blocks="absolute split of atm vs instrument",
        ),
        dict(
            limitation="Sonic-T (thermal turbulence) unusable",
            detail="median 0.084 K with exact zeros; |rho| < 0.08 with both wind and PSF",
            blocks="direct Cn^2 proxy",
        ),
        dict(
            limitation="Louver state is a night-level label",
            detail=f"{int((frac > 0.9).sum())}/{len(frac)} nights all-open; only "
            f"{len(mixed)} mixed nights",
            blocks="ventilation term g(V) — the §16c contrast",
        ),
    ]
).set_index("limitation")
display(lims)

### What would actually constrain $g(V,\Delta)$

Ordered by value per unit effort:

1. **Deliberate louver A/B switching within single nights.** Currently 67 of 73 nights are
   all-open, so ventilation is confounded with night. Toggling louvers on a
   ~30-minute cadence within nights converts this from a night-level label into a real
   experiment — by far the cheapest decisive change, and the only clean route to $g(V)$.
2. **Extend the mirror/air thermal join to full cadence and repair the sonic-T sensors.**
   $|\Delta T|$ is the one confirmed driver; the measured exponent $p\simeq0.5$ vs theory
   $6/5$ is most likely proxy attenuation, so better-placed thermal sensors (in the light
   path, not just on the glass) should steepen it toward theory and sharpen $C$.
3. **Accumulate high-wind nights.** No amount of reanalysis fixes a 17.5 m/s ceiling; any
   20 m/s statement stays extrapolation.
4. **Recover a zenith-corrected seeing reference** (fix `seeing_zenith_500nm`, or use a
   MASS/DIMM turbulence profile) to let $s$ be fit rather than assumed.

### Guidance for reusing the sibling notebooks

* Do **not** quadrature-subtract a noisy atmospheric proxy per exposure — fit the budget jointly.
* Check the **sign** of any control's correlation before calling it a confound; a negative
  $r_{xz}$ makes partialling inflate, not deflate (§3a).
* Report **within-night** estimates as primary, pooled as diagnostic only.
* Cluster standard errors on night, and run the **placebo** on every headline number.

> **Recommended follow-up:** annotate §16b/§16c of
> `Wind_Loading_and_Ingression_Operational_Limits.ipynb` with the suppressor-variable and
> clipping-artifact caveats, and mark the 20 m/s degradation figures as superseded by the
> upper limit derived in §4 here, so they are not carried into operational limits.